In [1]:
# 1. interpolation w.r.t. n instead of e
# 2. added semi-analytical calculation

import tmm
import tmm_helper as tmm_h

from numpy import pi, inf
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
%matplotlib inline
#%matplotlib qt
#plt.rcParams['figure.figsize'] = [3,2]


# Load external plotting functions
from plot_functions import plot_setup, plot, legend

# Load plotting colors
import colors # make available colors from schmid_colors.py 

# Image file settings
fmt = '.png' # image format (use png for PowerPoint, pdf and eps for publications)
dpi = 300 # image resolution, density of pixels per inch (use at least 300)
fig_dir = 'C:\\Users\\kl89\\MS Window Project\\Figures\\'

degree = pi/180

In [20]:
def generate_StackofStacks(gam, a, nb, num_stacks, t_prop):
    n_list = []
    d_list = []
    c_list = []
    c_list_KK = []

    n_list_KK, d_list_KK = tmm_h.generate_n_and_d_new(gam, a, nb, delta=0.03, plot_flag=False)
    t_KK = np.sum(d_list_KK)
    t_inc = t_prop*t_KK
    #print(f"t_KK is: {t_KK}")
    #print(f"t_inc is: {t_inc}")
    t_total = num_stacks*(t_inc + t_KK)
    #print(f"t_total is: {t_total}")
    #print(f'length of KK MS is: {t_KK}')

    k_bulk = np.sum(d_list_KK * np.imag(n_list_KK)) / t_KK
    #print(f'k_bulk from np.sum is: {k_bulk}')
    losses_total = t_total * k_bulk
    #print(f'total losses are: {losses_total}')

    for i in range(len(d_list_KK)):
        c_list_KK.append('c')

    for i in range(num_stacks):
        n_list.extend(n_list_KK)
        d_list.extend(d_list_KK)
        c_list.extend(c_list_KK)
        n_list.append(nb)
        d_list.append(t_inc)
        c_list.append('i')
    
    d_list.append(inf)
    d_list.insert(0, inf)
    n_list.append(nb)
    n_list.insert(0, nb)
    c_list.insert(0, 'i')
    c_list.append('i')


    return (n_list, d_list, c_list, losses_total)



In [ ]:
# calculate TRA of stack of stacks

#A = 0.00178063
A = 5
gam = 0.001
nb = 1.7
lamb = 3
num_stacks = 2
t_prop = 0

n_list_KK, d_list_KK = tmm_h.generate_n_and_d_new(gam, A, nb, delta=0.03, plot_flag=False)
losses_KK = np.sum(d_list_KK * np.imag(n_list_KK))
trans_KK_bulk = np.exp(-4*np.pi*losses_KK/lamb)
emiss_KK_bulk = 1 - trans_KK_bulk
c_list_KK = []
for i in range(len(d_list_KK)):
    c_list_KK.append('c')

d_list_KK.append(inf)
d_list_KK.insert(0, inf)
n_list_KK.append(nb)
n_list_KK.insert(0, nb)
c_list_KK.insert(0, 'i')
c_list_KK.append('i')

n_list_reversed_KK = n_list_KK[::-1]
d_list_reversed_KK = d_list_KK[::-1]
c_list_reversed_KK = c_list_KK[::-1]


T_LR_KK, R_LR_KK, A_LR_KK = tmm_h.TRA_inc(n_list_KK, d_list_KK, c_list_KK)
T_RL_KK, R_RL_KK, A_RL_KK = tmm_h.TRA_inc(n_list_reversed_KK, d_list_reversed_KK, c_list_reversed_KK)

print(f'T_LR_KK: {T_LR_KK}')
print(f'R_LR_KK: {R_LR_KK}')
print(f'R_RL_KK: {R_RL_KK}')
print(f'A_LR_KK: {A_LR_KK}')
print(f'A_RL_KK: {A_RL_KK}')
print(f'alpha:  {4*np.pi*losses_KK/lamb}')

for n in range(1,num_stacks+1):
    print('')
    print(f'-------{n} stacks--------')
    n_list, d_list, c_list, losses_total = generate_StackofStacks(gam, A, nb, n, t_prop)

    n_list_reversed = n_list[::-1]
    d_list_reversed = d_list[::-1]
    c_list_reversed = c_list[::-1]

    T_LR, R_LR, A_LR = tmm_h.TRA_inc(n_list, d_list, c_list)
    T_RL, R_RL, A_RL = tmm_h.TRA_inc(n_list_reversed, d_list_reversed, c_list_reversed)
    
    T_analytic = T_RL_KK**n
    geom_series = 0
    for k in range(n):
        geom_series = geom_series + T_RL_KK**k
    
    geom_series_2 = 0
    for l in range(n-1):
        inner_geom = 0
        for m in range(l+1):
            inner_geom = inner_geom + T_RL_KK**(2*m)
        geom_series_2 = geom_series_2 + inner_geom*T_RL_KK**(n-l-1)
    A_RL_analytic = A_RL_KK*geom_series + A_LR_KK*R_RL_KK*geom_series_2
    #A_LR_analytic = A_LR_KK*geom_series
    '''
    if n==3:
        A_RL_analytic = A_RL_KK + (A_RL_KK + A_LR_KK*R_RL_KK)*geom_series + A_LR_KK*R_RL_KK*T_RL_KK**3
    elif n==4:
        A_RL_analytic = A_RL_KK + (A_RL_KK + A_LR_KK*R_RL_KK)*geom_series + A_LR_KK*R_RL_KK*T_RL_KK**3 + \
            A_LR_KK*R_RL_KK*T_RL_KK**4 + A_LR_KK*R_RL_KK*T_RL_KK**5
    else:
        A_RL_analytic = A_RL_KK + (A_RL_KK + A_LR_KK*R_RL_KK)*geom_series
    '''
    print(f'T_RL: {T_RL}')
    print(f'T_analytic: {T_analytic}')
    print(f'A_RL: {A_RL}')
    print(f'A_RL_analytic: {A_RL_analytic}')
    print(f'FOM: {T_RL**2/A_RL}')
    print(f'FOM analytic: {T_analytic**2/A_RL_analytic}')

    del_T = T_analytic - T_RL
    del_A = A_RL_analytic - A_RL

    if n > 1:                   # start comparing slopes at N=2
        print(f"N={n:3d} ΔT={del_T:+.3e}, ΔA={del_A:+.3e}  (ΔA per slab ≈ {(del_A - delA_prev):+.3e})")

    delA_prev = del_A

trans_bulk = np.exp(-4*np.pi*losses_total/lamb)
emiss_bulk = 1 - trans_bulk

#print(R_list_RL_KK)

#print(A_RL_analytic)


Number of Layers: 128
T_LR_KK: 0.9808332686181755
R_LR_KK: 3.2420148838042454e-06
R_RL_KK: 0.00030060801840422
A_LR_KK: 0.01916348936694065
A_RL_KK: 0.01886612336341568
alpha:  0.019384558947170817

-------1 stacks--------
Number of Layers: 128
T_RL: 0.9808332686181801
T_analytic: 0.9808332686181801
A_RL: 0.01886612336341568
A_RL_analytic: 0.01886612336341568
FOM: 50.99266459233246
FOM analytic: 50.99266459233246

-------2 stacks--------
Number of Layers: 128
T_RL: 0.9620339017657978
T_analytic: 0.962033900828223
A_RL: 0.03737629511095045
A_RL_analytic: 0.03737629509291085
FOM: 24.761930667536134
FOM analytic: 24.761930631222686
N=  2 ΔT=-9.376e-10, ΔA=-1.804e-11  (ΔA per slab ≈ -1.804e-11)

-------3 stacks--------
Number of Layers: 128
T_RL: 0.9435948581947439
T_analytic: 0.9435948554708441
A_RL: 0.05553712318638442
A_RL_analytic: 0.05553712309856463
FOM: 16.032001755356383
FOM analytic: 16.03200168814749
N=  3 ΔT=-2.724e-09, ΔA=-8.782e-11  (ΔA per slab ≈ -6.978e-11)

-------4 stacks-

In [ ]:
# plotting RTA forwards and backwards

savename = f'Abs_diff_A={A}_x0={gam}_N={num_stacks}'
xlabel = 'Wavelength ($\mu$m)'; ylabel = 'Fraction of Power'
title = f'Error in Absorbance (A={A}, x$_0$={gam}, N={num_stacks})'
fig,ax = plot_setup(xlabel,ylabel,title=title,xmin=lambda_list[0],xmax=lambda_list[-1],xstep=1,figsize=(5,4),auto_scale=True)
#plt.rcParams['figure.figsize'] = [2,1]
plot(fig,ax,lambda_list,T_list_RL,label=r'T$_{RL}$',color=colors.blue,auto_scale=True)
plot(fig,ax,lambda_list,A_list_RL,label=r'A$_{RL}$',color=colors.red,auto_scale=True)
#plot(fig,ax,lambda_list,T_list_analytic, '--', label=r'T$_{analytic}$',color=colors.light_blue,auto_scale=True)
#plot(fig,ax,lambda_list,A_list_RL_analytic,label=r'A$_{analytic}$',color=colors.red,auto_scale=True)
#plot(fig,ax,lambda_list,trans_bulk, '--', label=r'trans_bulk',color=colors.red,auto_scale=True)
small_term = A_list_LR_KK*R_list_RL_KK*T_list_LR_KK
#plot(fig,ax,lambda_list,A_list_RL-A_list_RL_analytic,label=r'A$_{{TMM}}$ - A$_{{analytic}}$',color=colors.blue,auto_scale=True)
#plot(fig,ax,lambda_list,small_term, '--', label=r'A$_{LR}$R$_{RL}$T',color=colors.light_red,auto_scale=True)
#plot(fig,ax,lambda_list,A_list_RL_analytic, '--', label=r'A$_{RL,analytic}$',color=colors.light_red,auto_scale=True)
#plot(fig,ax,lambda_list,emiss_bulk, '--', label=r'emiss_bulk',color=colors.light_purple,auto_scale=True)

#plot(fig,ax,lambda_list,R_list_RL,label=r'R$_{RL}$',color=colors.green,auto_scale=True)
#plot(fig,ax,lambda_list,R_list_LR, '--', label=r'R$_{LR}$',color=colors.light_purple,auto_scale=True)

plt.savefig(fig_dir+savename+fmt,bbox_inches='tight',transparent=False,dpi=dpi) # save figure as image file

# Create and save legend for above figure
legend(fig,ax,auto_scale=True) # create legend from curves labeled above
plt.savefig(fig_dir+savename+'legend'+fmt, bbox_inches='tight', transparent=True, dpi=dpi) # save legend as image file


'''
plt.figure()
#plt.plot(lambda_list,T_list_LR, label='T_LR')
#print(T_list_LR/trans_bulk)
plt.plot(lambda_list,T_list_RL, label='T_RL')
plt.plot(lambda_list,T_list_analytic, '--', label='T_analytic')
#plt.plot(lambda_list,T_list_LR_KK, label='T_LR_KK')
#plt.plot(lambda_list,R_list_LR, label='R_LR')
#plt.plot(lambda_list,A_list_LR, label='A_LR')
#plt.plot(lambda_list,R_list_RL, label='R_RL')
plt.plot(lambda_list,A_list_RL, label='A_RL')
plt.plot(lambda_list,A_list_RL_analytic, '--', label='A_RL_analytic')
#plt.plot(lambda_list,A_list_LR_analytic, '--', label='A_LR_analytic')
#plt.plot(lambda_list,trans_bulk, '--', label='T_bulk')
#plt.plot(lambda_list,emiss_bulk, '--', label='A_bulk')
plt.xlabel('Wavelength ($\mu$m)')
plt.ylabel('Fraction of Power')
plt.title(f'Transmittance and Emittance for N={num_stacks}')
plt.legend(loc='center right')
'''

In [18]:
print(f'T_LR: {T_LR}')
print(f'T_RL: {T_RL}')
print(f'A_RL: {A_RL}')
print(f'T_analytic: {T_analytic}')
print(f'A_RL_analytic: {A_RL_analytic}')

T_LR: 0.6790532428523582
T_RL: 0.6790532428524204
A_RL: 0.3166799580969233
T_analytic: 0.6790531416459511
A_RL_analytic: 0.31600388018675696


In [16]:
# calculate FOM_KK and equivalently lossy bulk

FOM_LR = T_LR**2 / A_LR
FOM_RL = T_RL**2 / A_RL
FOM_RL_analytic = T_analytic**2 / A_RL_analytic
FOM_bulk = trans_bulk**2 / emiss_bulk

FOM_RL_KK = T_RL_KK**2 / A_RL_KK
FOM_bulk_KK = trans_KK_bulk**2 / emiss_KK_bulk
print(f'FOM for 1 KK: {FOM_RL_KK}')
print(f'FOM for 1 bulk: {FOM_bulk_KK}')
print(f'FOM enhancement for 1: {FOM_RL_KK/FOM_bulk_KK}')
print(f'FOM for {num_stacks} KK: {FOM_RL}')
print(f'FOM for {num_stacks} bulk: {FOM_bulk}')
print(f'FOM enhancement for {num_stacks}: {FOM_RL/FOM_bulk}')
print(f'FOM for {num_stacks} KK analytic: {FOM_RL_analytic}')
print(f'FOM enhancement for {num_stacks} analytic: {FOM_RL_analytic/FOM_bulk}')

FOM for 1 KK: 50.99266459233246
FOM for 1 bulk: 50.10826503818407
FOM enhancement for 1: 1.0176497740138168
FOM for 2 KK: 24.761930667536134
FOM for 2 bulk: 24.334983784018732
FOM enhancement for 2: 1.0175445723451761
FOM for 2 KK analytic: 24.761930631222686
FOM enhancement for 2 analytic: 1.0175445708529438
